In [ ]:
## pip install ase

  Using cached ase-3.26.0-py3-none-any.whl.metadata (4.1 kB)
Using cached ase-3.26.0-py3-none-any.whl (2.9 MB)
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


## 测试更换ASE生成的POSCAR文件的首行注释为化学式（即标注出每种原子的个数）

In [1]:
import os
import pandas as pd
from mp_api.client import MPRester
from ase.io import read, write

def read_compound_list(file_path):
    """
    从表格文件中读取化合物名称
    支持CSV和Excel格式(.csv, .xlsx, .xls)
    
    参数:
        file_path: 表格文件路径
    
    返回:
        化合物名称列表
    """
    file_ext = os.path.splitext(file_path)[1].lower()
    
    try:
        if file_ext == '.csv':
            df = pd.read_csv(file_path)
        elif file_ext in ['.xlsx', '.xls']:
            df = pd.read_excel(file_path)
        else:
            raise ValueError(f"不支持的文件格式: {file_ext}. 请使用 .csv, .xlsx 或 .xls 文件")
        
        # 尝试找到包含化合物名称的列
        # 优先查找常见的列名
        possible_columns = ['compound',  'formula' ,'name',
                          '化合物',  '分子式']
        
        compound_column = None
        for col in df.columns:
            if col.lower() in [c.lower() for c in possible_columns]:
                compound_column = col
                break
        
        # 如果没找到，使用第一列
        if compound_column is None:
            compound_column = df.columns[0]
            print(f"未找到标准列名，使用第一列: '{compound_column}'")
        else:
            print(f"使用列: '{compound_column}'")
        
        # 提取化合物名称，去除空值和重复项
        compounds = df[compound_column].dropna().astype(str).str.strip().unique().tolist()
        
        print(f"从文件中读取到 {len(compounds)} 个化合物")
        return compounds
        
    except Exception as e:
        print(f"读取文件时出错: {str(e)}")
        raise


def download_structures_from_mp(compound_names, api_key, output_dir="foound ID 89-cif"):
    """
    根据化合物名称列表从Materials Project下载CIF文件
    
    参数:
        compound_names: 化合物名称列表
        api_key: Materials Project的API密钥
        output_dir: CIF文件保存目录
    
    返回:
        包含化合物名称和MP ID的DataFrame
    """
    
    # 创建存储CIF文件的目录
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # 存储结果
    results = []
    
    # 初始化MP API客户端
    with MPRester(api_key) as mpr:
        total = len(compound_names)
        for idx, compound in enumerate(compound_names, 1):
            try:
                print(f"\n[{idx}/{total}] 正在搜索 {compound}...")
                
                # 搜索化合物
                docs = mpr.materials.summary.search(
                    formula=compound,
                    fields=["material_id", "formula_pretty", "structure","efermi", "energy_above_hull"]
                )
                
                if not docs:
                    print(f"  ❌ 未找到 {compound} 的数据")
                    results.append({
                        'compound_name': compound,
                        'mp_id': 'nan',
                        'formula': '',
                        'efermi':'',
                        'energy_above_hull': '',
                        'cif_file': '',
                        'status': 'Not Found'
                    })
                    continue
                
                # 按能量排序（优先下载最稳定的结构）
                docs = sorted(docs, key=lambda x: x.energy_above_hull if x.energy_above_hull else float('inf'))
                
                # 遍历所有找到的结构
                for doc_idx, doc in enumerate(docs):
                    mp_id = doc.material_id
                    formula = doc.formula_pretty
                    efermi = doc.efermi,
                    structure = doc.structure
                    energy = doc.energy_above_hull if doc.energy_above_hull else 0
                    composition = structure.composition

                    
                    stability = "⭐ 稳定" if energy < 0.01 else f"能量差: {energy:.3f} eV/atom"
                    print(f"  ✓ 找到: {mp_id} ({formula}) - {stability}")
                    
                    # 保存CIF文件
                    cif_filename = os.path.join(output_dir, f"{mp_id}_{compound.replace('/', '-')}.cif")
                    poscar_filename = os.path.join(output_dir, f"{mp_id}_{compound.replace('/', '-')}.vasp")
                    structure.to(filename=cif_filename, fmt="cif")
                    print(f"    已保存: {cif_filename}")
                    # 使用ASE读取刚生成的CIF文件，然后写入POSCAR格式
                    atoms = read(cif_filename)        # 读取CIF文件
                    base_name = os.path.splitext(cif_filename)[0]  # 获取不带后缀的文件名
                    print(base_name)
                    atoms.comment = base_name  # 使用文件名作为首行注释
                    write(poscar_filename, atoms, format='vasp',direct=True)  # 写入VASP格式
                    print(f"    已转换并保存: {poscar_filename}")
                    # 记录结果
                    results.append({
                        'compound_name': compound,
                        'mp_id': mp_id,
                        'formula': formula,
                        'efermi': efermi,
                        'composition': str(composition),
                        'cif_file': cif_filename,
                        'status': 'Success'
                    })
                    
            except Exception as e:
                print(f"  ❌ 处理 {compound} 时出错: {str(e)}")
                results.append({
                    'compound_name': compound,
                    'mp_id': 'Error',
                    'formula': '',
                    'efermi':'',
                    'energy_above_hull': '',
                    'composition': '',
                    'cif_file': '',
                    'status': f'Error: {str(e)}'
                })
    
    # 创建DataFrame
    df = pd.DataFrame(results)
    
    # 保存到CSV文件
    output_csv = 'found ID-89.csv'
    df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"\n{'='*60}")
    print(f"✓ 总结信息已保存到: {output_csv}")
    print(f"✓ CIF文件已保存到: {output_dir}/")
    print(f"{'='*60}")
    
    return df


def main(input_file, api_key):
    """
    主函数：从表格文件读取化合物并下载结构
    
    参数:
        input_file: 输入的表格文件路径
        api_key: Materials Project API密钥
    """
    print("="*60)
    print("Materials Project 批量下载工具")
    print("="*60)
    
    # 读取化合物列表
    print(f"\n正在读取文件: {input_file}")
    compounds = read_compound_list(input_file)
    
    if not compounds:
        print("错误：未能从文件中读取到任何化合物")
        return
    
    print(f"\n将要搜索的化合物:")
    for i, comp in enumerate(compounds, 1):
        print(f"  {i}. {comp}")
    
    # 下载结构
    print(f"\n开始从Materials Project下载...")
    results_df = download_structures_from_mp(compounds, api_key)
    
    # 显示统计信息
    print("\n" + "="*60)
    print("下载完成！统计信息:")
    print("="*60)
    success_count = len(results_df[results_df['status'] == 'Success'])
    not_found_count = len(results_df[results_df['status'] == 'Not Found'])
    error_count = len(results_df[results_df['status'].str.contains('Error', na=False)])
    
    print(f"✓ 成功下载: {success_count}")
    print(f"❌ 未找到: {not_found_count}")
    print(f"⚠ 错误: {error_count}")
    print(f"总计: {len(compounds)} 个化合物")
    
    # 显示结果摘要
    print("\n结果摘要 (化合物名称 -> MP ID):")
    print("-"*60)
    summary = results_df.groupby('compound_name')['mp_id'].apply(lambda x: ', '.join(x.astype(str))).to_dict()
    for compound, mp_ids in summary.items():
        print(f"{compound}: {mp_ids}")
    
    return results_df


# 使用示例
if __name__ == "__main__":
    # ====== 配置区域 ======
    
    # 你的Materials Project API密钥
    # 从这里获取: https://next-gen.materialsproject.org/api
    API_KEY = "YDSxOUAtBvuvOYGI8UauQJjRetrwc5ss"
    
    # 输入文件路径（支持 .csv, .xlsx, .xls）
    INPUT_FILE = "find ID-89.xlsx"  # 或 "compounds.csv"
    
    # =====================
    
    # 运行主程序
    try:
        results = main(INPUT_FILE, API_KEY)
        print("\n程序执行完成！")
    except Exception as e:
        print(f"\n程序执行出错: {str(e)}")


Materials Project 批量下载工具

正在读取文件: find ID-89.xlsx
使用列: 'name'
从文件中读取到 89 个化合物

将要搜索的化合物:
  1. CuMo
  2. CuOs
  3. Cu2W
  4. Cu2Os
  5. Cu3W
  6. Cu4W
  7. CoOs
  8. Co2Os
  9. Co3Os
  10. Co4W
  11. Co4Os
  12. RuBe
  13. RuRh
  14. Ru2Be
  15. Ru2Co
  16. Ru2Ni
  17. Ru2Cu
  18. Ru2Rh
  19. Ru2Pd
  20. Ru2Os
  21. Ru2Ir
  22. Ru3Be
  23. Ru3Al
  24. Ru3Cr
  25. Ru3Co
  26. Ru3Ni
  27. Ru3Cu
  28. Ru3Rh
  29. Ru3Pd
  30. Ru3Os
  31. Ru3Ir
  32. Ru4Be
  33. Ru4Mg
  34. Ru4Al
  35. Ru4Cr
  36. Ru4Co
  37. Ru4Ni
  38. Ru4Cu
  39. Ru4Zn
  40. Ru4Rh
  41. Ru4Pd
  42. Ru4Os
  43. Ru4Ir
  44. Ru4Pt
  45. Ru4V
  46. RhOs
  47. Rh2Mo
  48. Rh2Ru
  49. Rh2Os
  50. Rh2Ir
  51. Rh3Nb
  52. Rh3Mo
  53. Rh3Ru
  54. Rh3Os
  55. Rh3Ir
  56. Rh4Nb
  57. Rh4Mo
  58. Rh4Ru
  59. Rh4Os
  60. Rh4Ir
  61. IrBe
  62. IrMo
  63. IrPd
  64. IrOs
  65. IrV
  66. IrTi
  67. Ir2Co
  68. Ir2Nb
  69. Ir2Mo
  70. Ir2Ru
  71. Ir2Rh
  72. Ir2Pd
  73. Ir2Os
  74. Ir2V
  75. Ir2Ti
  76. Ir3Nb
  77. Ir3Mo
  78. Ir3Ru
  7

Retrieving SummaryDoc documents: 0it [00:00, ?it/s]

  ❌ 未找到 CuMo 的数据

[2/89] 正在搜索 CuOs...


Retrieving SummaryDoc documents: 0it [00:00, ?it/s]

  ❌ 未找到 CuOs 的数据

[3/89] 正在搜索 Cu2W...


...

  ❌ 未找到 Ir4In 的数据

[88/89] 正在搜索 Ir4W...


Retrieving SummaryDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ 找到: mp-1223666 (Ir4W) - 能量差: 0.232 eV/atom
    已保存: foound ID 89-cif\mp-1223666_Ir4W.cif
foound ID 89-cif\mp-1223666_Ir4W
    已转换并保存: foound ID 89-cif\mp-1223666_Ir4W.vasp

[89/89] 正在搜索 Ir4Os...


Retrieving SummaryDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ 找到: mp-1223677 (Ir4Os) - 能量差: 0.027 eV/atom
    已保存: foound ID 89-cif\mp-1223677_Ir4Os.cif
foound ID 89-cif\mp-1223677_Ir4Os
    已转换并保存: foound ID 89-cif\mp-1223677_Ir4Os.vasp

✓ 总结信息已保存到: found ID-89.csv
✓ CIF文件已保存到: foound ID 89-cif/

下载完成！统计信息:
✓ 成功下载: 46
❌ 未找到: 58
⚠ 错误: 0
总计: 89 个化合物

结果摘要 (化合物名称 -> MP ID):
------------------------------------------------------------
Co2Os: nan
Co3Os: mp-1183718
Co4Os: mp-1226016
Co4W: nan
CoOs: mp-1226000
Cu2Os: nan
Cu2W: nan
Cu3W: mp-1183970
Cu4W: nan
CuMo: nan
CuOs: nan
Ir2Co: nan
Ir2Mo: nan
Ir2Nb: nan
Ir2Os: nan
Ir2Pd: nan
Ir2Rh: nan
Ir2Ru: nan
Ir2Ti: nan
Ir2V: nan
Ir3In: nan
Ir3Mo: mp-11482
Ir3Nb: mp-1339, mp-1186231
Ir3Os: mp-1184761
Ir3Pd: nan
Ir3Ru: mp-974358
Ir3Ti: mp-1089
Ir3W: mp-30745
Ir4In: nan
Ir4Mo: mp-1221426
Ir4Nb: nan
Ir4Os: mp-1223677
Ir4Pd: nan
Ir4W: mp-1223666
IrBe: nan
IrMo: mp-1221414, mp-11481
IrOs: mp-1223655
IrPd: mp-1223675
IrTi: mp-12594, mp-1235
IrV: mp-982070, mp-1178810, mp-1178808, mp-1079582, mp-1204410, mp-1079

## 使用structure.to的形式生成POSCAR文件，出现-0.0的形式，更换使用上面的ASE格式

In [2]:
import os
import pandas as pd
from mp_api.client import MPRester
#from ase.io import read, write

def read_compound_list(file_path):
    """
    从表格文件中读取化合物名称
    支持CSV和Excel格式(.csv, .xlsx, .xls)
    
    参数:
        file_path: 表格文件路径
    
    返回:
        化合物名称列表
    """
    file_ext = os.path.splitext(file_path)[1].lower()
    
    try:
        if file_ext == '.csv':
            df = pd.read_csv(file_path)
        elif file_ext in ['.xlsx', '.xls']:
            df = pd.read_excel(file_path)
        else:
            raise ValueError(f"不支持的文件格式: {file_ext}. 请使用 .csv, .xlsx 或 .xls 文件")
        
        # 尝试找到包含化合物名称的列
        # 优先查找常见的列名
        possible_columns = ['compound',  'formula' ,'name',
                          '化合物',  '分子式']
        
        compound_column = None
        for col in df.columns:
            if col.lower() in [c.lower() for c in possible_columns]:
                compound_column = col
                break
        
        # 如果没找到，使用第一列
        if compound_column is None:
            compound_column = df.columns[0]
            print(f"未找到标准列名，使用第一列: '{compound_column}'")
        else:
            print(f"使用列: '{compound_column}'")
        
        # 提取化合物名称，去除空值和重复项
        compounds = df[compound_column].dropna().astype(str).str.strip().unique().tolist()
        
        print(f"从文件中读取到 {len(compounds)} 个化合物")
        return compounds
        
    except Exception as e:
        print(f"读取文件时出错: {str(e)}")
        raise


def download_structures_from_mp(compound_names, api_key, output_dir="foound ID 89"):
    """
    根据化合物名称列表从Materials Project下载CIF文件
    
    参数:
        compound_names: 化合物名称列表
        api_key: Materials Project的API密钥
        output_dir: CIF文件保存目录
    
    返回:
        包含化合物名称和MP ID的DataFrame
    """
    
    # 创建存储CIF文件的目录
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # 存储结果
    results = []
    
    # 初始化MP API客户端
    with MPRester(api_key) as mpr:
        total = len(compound_names)
        for idx, compound in enumerate(compound_names, 1):
            try:
                print(f"\n[{idx}/{total}] 正在搜索 {compound}...")
                
                # 搜索化合物
                docs = mpr.materials.summary.search(
                    formula=compound,
                    fields=["material_id", "formula_pretty", "structure","efermi", "energy_above_hull"]
                )
                
                if not docs:
                    print(f"  ❌ 未找到 {compound} 的数据")
                    results.append({
                        'compound_name': compound,
                        'mp_id': 'nan',
                        
                        'formula': '',
                        'efermi':'',
                        'energy_above_hull': '',
                        'cif_file': '',
                        'status': 'Not Found'
                    })
                    continue
                
                # 按能量排序（优先下载最稳定的结构）
                docs = sorted(docs, key=lambda x: x.energy_above_hull if x.energy_above_hull else float('inf'))
                
                # 遍历所有找到的结构
                for doc_idx, doc in enumerate(docs):
                    mp_id = doc.material_id
                    formula = doc.formula_pretty
                    efermi = doc.efermi,
                    structure = doc.structure
                    energy = doc.energy_above_hull if doc.energy_above_hull else 0
                    
                    stability = "⭐ 稳定" if energy < 0.01 else f"能量差: {energy:.3f} eV/atom"
                    print(f"  ✓ 找到: {mp_id} ({formula}) - {stability}")
                    
                    # 保存CIF文件
                    cif_filename = os.path.join(output_dir, f"{mp_id}_{compound.replace('/', '-')}.cif")
                    poscar_filename = os.path.join(output_dir, f"{mp_id}_{compound.replace('/', '-')}.vasp")
                    structure.to(filename=cif_filename, fmt="cif")
                    print(f"    已保存: {cif_filename}")
                    structure.to(filename=poscar_filename, fmt="poscar")  # 使用 'poscar' 或 'vasp' 作为格式标识
                    print(f"    已保存: {poscar_filename}")
                    # 记录结果
                    results.append({
                        'compound_name': compound,
                        'mp_id': mp_id,
                        'formula': formula,
                        'efermi': efermi,
                        'energy_above_hull': f"{energy:.4f}" if energy else "0",
                        'cif_file': cif_filename,
                        'status': 'Success'
                    })
                    
            except Exception as e:
                print(f"  ❌ 处理 {compound} 时出错: {str(e)}")
                results.append({
                    'compound_name': compound,
                    'mp_id': 'Error',
                   
                    'formula': '',
                    'efermi':'',
                    'energy_above_hull': '',
                    'cif_file': '',
                    'status': f'Error: {str(e)}'
                })
    
    # 创建DataFrame
    df = pd.DataFrame(results)
    
    # 保存到CSV文件
    output_csv = 'found ID-89.csv'
    df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"\n{'='*60}")
    print(f"✓ 总结信息已保存到: {output_csv}")
    print(f"✓ CIF文件已保存到: {output_dir}/")
    print(f"{'='*60}")
    
    return df


def main(input_file, api_key):
    """
    主函数：从表格文件读取化合物并下载结构
    
    参数:
        input_file: 输入的表格文件路径
        api_key: Materials Project API密钥
    """
    print("="*60)
    print("Materials Project 批量下载工具")
    print("="*60)
    
    # 读取化合物列表
    print(f"\n正在读取文件: {input_file}")
    compounds = read_compound_list(input_file)
    
    if not compounds:
        print("错误：未能从文件中读取到任何化合物")
        return
    
    print(f"\n将要搜索的化合物:")
    for i, comp in enumerate(compounds, 1):
        print(f"  {i}. {comp}")
    
    # 下载结构
    print(f"\n开始从Materials Project下载...")
    results_df = download_structures_from_mp(compounds, api_key)
    
    # 显示统计信息
    print("\n" + "="*60)
    print("下载完成！统计信息:")
    print("="*60)
    success_count = len(results_df[results_df['status'] == 'Success'])
    not_found_count = len(results_df[results_df['status'] == 'Not Found'])
    error_count = len(results_df[results_df['status'].str.contains('Error', na=False)])
    
    print(f"✓ 成功下载: {success_count}")
    print(f"❌ 未找到: {not_found_count}")
    print(f"⚠ 错误: {error_count}")
    print(f"总计: {len(compounds)} 个化合物")
    
    # 显示结果摘要
    print("\n结果摘要 (化合物名称 -> MP ID):")
    print("-"*60)
    summary = results_df.groupby('compound_name')['mp_id'].apply(lambda x: ', '.join(x.astype(str))).to_dict()
    for compound, mp_ids in summary.items():
        print(f"{compound}: {mp_ids}")
    
    return results_df


# 使用示例
if __name__ == "__main__":
    # ====== 配置区域 ======
    
    # 你的Materials Project API密钥
    # 从这里获取: https://next-gen.materialsproject.org/api
    API_KEY = "YDSxOUAtBvuvOYGI8UauQJjRetrwc5ss"
    
    # 输入文件路径（支持 .csv, .xlsx, .xls）
    INPUT_FILE = "find ID-89.xlsx"  # 或 "compounds.csv"
    
    # =====================
    
    # 运行主程序
    try:
        results = main(INPUT_FILE, API_KEY)
        print("\n程序执行完成！")
    except Exception as e:
        print(f"\n程序执行出错: {str(e)}")


Materials Project 批量下载工具

正在读取文件: find ID-89.xlsx
使用列: 'name'
从文件中读取到 89 个化合物

将要搜索的化合物:
  1. CuMo
  2. CuOs
  3. Cu2W
  4. Cu2Os
  5. Cu3W
  6. Cu4W
  7. CoOs
  8. Co2Os
  9. Co3Os
  10. Co4W
  11. Co4Os
  12. RuBe
  13. RuRh
  14. Ru2Be
  15. Ru2Co
  16. Ru2Ni
  17. Ru2Cu
  18. Ru2Rh
  19. Ru2Pd
  20. Ru2Os
  21. Ru2Ir
  22. Ru3Be
  23. Ru3Al
  24. Ru3Cr
  25. Ru3Co
  26. Ru3Ni
  27. Ru3Cu
  28. Ru3Rh
  29. Ru3Pd
  30. Ru3Os
  31. Ru3Ir
  32. Ru4Be
  33. Ru4Mg
  34. Ru4Al
  35. Ru4Cr
  36. Ru4Co
  37. Ru4Ni
  38. Ru4Cu
  39. Ru4Zn
  40. Ru4Rh
  41. Ru4Pd
  42. Ru4Os
  43. Ru4Ir
  44. Ru4Pt
  45. Ru4V
  46. RhOs
  47. Rh2Mo
  48. Rh2Ru
  49. Rh2Os
  50. Rh2Ir
  51. Rh3Nb
  52. Rh3Mo
  53. Rh3Ru
  54. Rh3Os
  55. Rh3Ir
  56. Rh4Nb
  57. Rh4Mo
  58. Rh4Ru
  59. Rh4Os
  60. Rh4Ir
  61. IrBe
  62. IrMo
  63. IrPd
  64. IrOs
  65. IrV
  66. IrTi
  67. Ir2Co
  68. Ir2Nb
  69. Ir2Mo
  70. Ir2Ru
  71. Ir2Rh
  72. Ir2Pd
  73. Ir2Os
  74. Ir2V
  75. Ir2Ti
  76. Ir3Nb
  77. Ir3Mo
  78. Ir3Ru
  7

Retrieving SummaryDoc documents: 0it [00:00, ?it/s]

  ❌ 未找到 CuMo 的数据

[2/89] 正在搜索 CuOs...


Retrieving SummaryDoc documents: 0it [00:00, ?it/s]

  ❌ 未找到 CuOs 的数据

[3/89] 正在搜索 Cu2W...


...

  ❌ 未找到 Ir4In 的数据

[88/89] 正在搜索 Ir4W...


Retrieving SummaryDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ 找到: mp-1223666 (Ir4W) - 能量差: 0.232 eV/atom
    已保存: foound ID 89\mp-1223666_Ir4W.cif
    已保存: foound ID 89\mp-1223666_Ir4W.vasp

[89/89] 正在搜索 Ir4Os...


Retrieving SummaryDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ 找到: mp-1223677 (Ir4Os) - 能量差: 0.027 eV/atom
    已保存: foound ID 89\mp-1223677_Ir4Os.cif
    已保存: foound ID 89\mp-1223677_Ir4Os.vasp

✓ 总结信息已保存到: found ID-89.csv
✓ CIF文件已保存到: foound ID 89/

下载完成！统计信息:
✓ 成功下载: 46
❌ 未找到: 58
⚠ 错误: 0
总计: 89 个化合物

结果摘要 (化合物名称 -> MP ID):
------------------------------------------------------------
Co2Os: nan
Co3Os: mp-1183718
Co4Os: mp-1226016
Co4W: nan
CoOs: mp-1226000
Cu2Os: nan
Cu2W: nan
Cu3W: mp-1183970
Cu4W: nan
CuMo: nan
CuOs: nan
Ir2Co: nan
Ir2Mo: nan
Ir2Nb: nan
Ir2Os: nan
Ir2Pd: nan
Ir2Rh: nan
Ir2Ru: nan
Ir2Ti: nan
Ir2V: nan
Ir3In: nan
Ir3Mo: mp-11482
Ir3Nb: mp-1339, mp-1186231
Ir3Os: mp-1184761
Ir3Pd: nan
Ir3Ru: mp-974358
Ir3Ti: mp-1089
Ir3W: mp-30745
Ir4In: nan
Ir4Mo: mp-1221426
Ir4Nb: nan
Ir4Os: mp-1223677
Ir4Pd: nan
Ir4W: mp-1223666
IrBe: nan
IrMo: mp-1221414, mp-11481
IrOs: mp-1223655
IrPd: mp-1223675
IrTi: mp-12594, mp-1235
IrV: mp-982070, mp-1178810, mp-1178808, mp-1079582, mp-1204410, mp-1079131, mp-1281, mp-569250
Rh2Ir: nan
Rh2Mo: nan
Rh2